
#1. Uniqueness: no duplicate rows in bridge tables
### Each pair of numbers should be equal.

In [0]:
%sql
-- 1) if primary key is unique
SELECT count(*), count(DISTINCT company_id)    FROM humanoid_robotics.gold.dim_company;

SELECT count(*), count(DISTINCT technology_id) FROM humanoid_robotics.gold.dim_technology;

SELECT count(*), count(DISTINCT event_id, company_id, role) FROM humanoid_robotics.gold.bridge_event_company;

SELECT count(*), count(DISTINCT event_id, technology_id)    FROM humanoid_robotics.gold.bridge_event_technology;

SELECT count(*), count(DISTINCT event_id, doc_id)           FROM humanoid_robotics.gold.bridge_event_source;


# 2. Referential integrity: every foreign key exists in the parent table
Expectations: All counts should be 0.

In [0]:
%sql
SELECT DISTINCT x.tech
FROM (
  SELECT explode(technology_areas) AS tech
  FROM humanoid_robotics.silver.events_extracted
) x
LEFT ANTI JOIN humanoid_robotics.gold.dim_technology d
  ON d.technology_id = x.tech;

SELECT count(*) FROM humanoid_robotics.gold.bridge_event_company b
LEFT ANTI JOIN humanoid_robotics.gold.fct_events f ON f.event_id = b.event_id;

SELECT count(*) FROM humanoid_robotics.gold.bridge_event_company b
LEFT ANTI JOIN humanoid_robotics.gold.dim_company c ON c.company_id = b.company_id;

SELECT count(*) FROM humanoid_robotics.gold.bridge_event_technology b
LEFT ANTI JOIN humanoid_robotics.gold.dim_technology t ON t.technology_id = b.technology_id;

SELECT count(*) FROM humanoid_robotics.gold.bridge_event_source b
LEFT ANTI JOIN humanoid_robotics.gold.dim_source s ON s.doc_id = b.doc_id;

# 3. Coverage: events without source, company or technology
Expectations: `no_source` must be 0. `no_company` and `no_technology` may be > 0 (e.g. policy events); record them in `prompt_issues.md`.

In [0]:
%sql
SELECT
  sum(CASE WHEN NOT EXISTS (SELECT 1 FROM humanoid_robotics.gold.bridge_event_source b     WHERE b.event_id = f.event_id) THEN 1 ELSE 0 END) AS no_source,
  sum(CASE WHEN NOT EXISTS (SELECT 1 FROM humanoid_robotics.gold.bridge_event_company b    WHERE b.event_id = f.event_id) THEN 1 ELSE 0 END) AS no_company,
  sum(CASE WHEN NOT EXISTS (SELECT 1 FROM humanoid_robotics.gold.bridge_event_technology b WHERE b.event_id = f.event_id) THEN 1 ELSE 0 END) AS no_technology
FROM humanoid_robotics.gold.fct_events f;

# 4. Business rule: funding events without a funded company
Expectations:  Funding events with no `target` or `issuer` role would be missed when summing funding by company. 0 rows.

In [0]:
%sql
SELECT f.event_id, f.amount, f.currency
FROM humanoid_robotics.gold.fct_events f
WHERE f.event_type = 'funding'
  AND NOT EXISTS (
    SELECT 1 FROM humanoid_robotics.gold.bridge_event_company b
    WHERE b.event_id = f.event_id AND b.role IN ('target', 'issuer')
  );